# read excel 

In [23]:
import pandas as pd
from pathlib import Path


def lees_evenementen(file_path):
    """
    Leest alle werkbladen uit het Excel-bestand
    en haalt de gegevens per rij op.

    Verwachte kolommen zijn ongeveer:
    Week | Soort content | Post content | Dag | Activiteit | Kanaal
    """

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"Excel-bestand niet gevonden: {file_path}"
        )

    # Lees alle werkbladen
    sheets = pd.read_excel(
        file_path,
        sheet_name=None,
        engine="openpyxl"
    )

    evenementen = []

    for sheet_name, df in sheets.items():

        # Kolomnamen opschonen
        df.columns = [
            str(column).strip().lower()
            for column in df.columns
        ]

        for _, row in df.iterrows():

            evenement = {
                "week": row.get("week"),
                "soort_content": row.get("soort content"),
                "post_content": row.get("post content"),
                "dag": row.get("dag"),
                "activiteit": row.get("activiteit"),
                "kanaal": row.get("kanaal"),
                "bron": sheet_name,
            }

            # Alleen regels toevoegen waar daadwerkelijk
            # een activiteit staat
            if pd.notna(evenement["activiteit"]):
                evenementen.append(evenement)

    return evenementen


if __name__ == "__main__":

    excel_file = (
        r"C:\Bodie\marketing code\calander_automatisering"
        r"\Jaarplanning social media 2026.xlsx"
    )

    evenementen = lees_evenementen(excel_file)

    print(f"Aantal evenementen: {len(evenementen)}")

    for evenement in evenementen:
        print(evenement)

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

# normalise data 

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime


class ExcelCalendarImporter:
    """
    Leest Excelbestanden met verschillende structuren.

    Per sheet kan handmatig worden aangegeven:
    - welke kolom de datum bevat
    - welke kolom de activiteit bevat

    De verschillende Excel-structuren worden vervolgens omgezet
    naar één standaard formaat:

    {
        "datum": datetime,
        "activiteit": str,
        "bron": str
    }
    """

    def __init__(self):
        self.excel_path = None
        self.sheets = {}
        self.mappings = {}
        self.events = []

    # ---------------------------------------------------------
    # 1. Excelbestand laden
    # ---------------------------------------------------------

    def load_excel(self, excel_path):
        """
        Leest alle sheets uit een Excelbestand.
        """

        self.excel_path = Path(excel_path)

        if not self.excel_path.exists():
            raise FileNotFoundError(
                f"Excelbestand niet gevonden: {self.excel_path}"
            )

        if self.excel_path.suffix.lower() not in [".xlsx", ".xls"\]:
            raise ValueError(
                "Alleen .xlsx en .xls bestanden worden ondersteund."
            )

        self.sheets = pd.read_excel(
            self.excel_path,
            sheet_name=None
        )

        print(f"\nExcelbestand geladen: {self.excel_path.name}")
        print(f"Aantal sheets: {len(self.sheets)}")

        for sheet_name, df in self.sheets.items():

            print(f"\nSheet: {sheet_name}")
            print("Kolommen:")

            for column in df.columns:
                print(f" - {column}")

        return list(self.sheets.keys())

    # ---------------------------------------------------------
    # 2. Beschikbare sheets ophalen
    # ---------------------------------------------------------

    def get_sheets(self):
        return list(self.sheets.keys())

    # ---------------------------------------------------------
    # 3. Kolommen van een sheet ophalen
    # ---------------------------------------------------------

    def get_columns(self, sheet_name):

        if sheet_name not in self.sheets:
            raise ValueError(
                f"Sheet '{sheet_name}' bestaat niet."
            )

        return list(self.sheets[sheet_name].columns)

    # ---------------------------------------------------------
    # 4. Voorbeelddata ophalen
    # ---------------------------------------------------------

    def get_preview(self, sheet_name, rows=5):

        if sheet_name not in self.sheets:
            raise ValueError(
                f"Sheet '{sheet_name}' bestaat niet."
            )

        return self.sheets[sheet_name].head(rows)

    # ---------------------------------------------------------
    # 5. Mapping instellen
    # ---------------------------------------------------------

    def set_mapping(
        self,
        sheet_name,
        date_column,
        activity_column
    ):
        """
        Geeft per sheet aan welke kolommen gebruikt moeten worden.

        Voorbeeld:

        importer.set_mapping(
            "Planning",
            date_column="Startdatum",
            activity_column="Werkzaamheid"
        )
        """

        if sheet_name not in self.sheets:
            raise ValueError(
                f"Sheet '{sheet_name}' bestaat niet."
            )

        df = self.sheets[sheet_name]

        if date_column not in df.columns:
            raise ValueError(
                f"Datumkolom '{date_column}' bestaat niet "
                f"in sheet '{sheet_name}'."
            )

        if activity_column not in df.columns:
            raise ValueError(
                f"Activiteitskolom '{activity_column}' bestaat niet "
                f"in sheet '{sheet_name}'."
            )

        self.mappings[sheet_name] = {
            "date_column": date_column,
            "activity_column": activity_column
        }

    # ---------------------------------------------------------
    # 6. Datum converteren
    # ---------------------------------------------------------

    @staticmethod
    def parse_date(value):
        """
        Probeert verschillende soorten Excel-datums
        om te zetten naar Python datetime.
        """

        if pd.isna(value):
            return None

        if isinstance(value, pd.Timestamp):
            return value.to_pydatetime()

        if isinstance(value, datetime):
            return value

        try:
            parsed = pd.to_datetime(
                value,
                dayfirst=True,
                errors="coerce"
            )

            if pd.isna(parsed):
                return None

            return parsed.to_pydatetime()

        except Exception:
            return None

    # ---------------------------------------------------------
    # 7. Eén sheet normaliseren
    # ---------------------------------------------------------

    def normalize_sheet(self, sheet_name):

        if sheet_name not in self.mappings:
            raise ValueError(
                f"Er is nog geen mapping ingesteld "
                f"voor sheet '{sheet_name}'."
            )

        df = self.sheets[sheet_name]

        mapping = self.mappings[sheet_name]

        date_column = mapping["date_column"]
        activity_column = mapping["activity_column"]

        events = []

        for index, row in df.iterrows():

            raw_date = row[date_column]
            raw_activity = row[activity_column]

            # Lege regels overslaan
            if pd.isna(raw_date) and pd.isna(raw_activity):
                continue

            # Datum converteren
            date = self.parse_date(raw_date)

            # Geen geldige datum -> regel overslaan
            if date is None:
                print(
                    f"Waarschuwing: regel {index + 2} "
                    f"in '{sheet_name}' heeft geen geldige datum."
                )
                continue

            # Geen activiteit -> regel overslaan
            if pd.isna(raw_activity):
                print(
                    f"Waarschuwing: regel {index + 2} "
                    f"in '{sheet_name}' heeft geen activiteit."
                )
                continue

            activity = str(raw_activity).strip()

            if not activity:
                continue

            event = {
                "datum": date,
                "activiteit": activity,
                "bron": sheet_name
            }

            events.append(event)

        return events

    # ---------------------------------------------------------
    # 8. Alle ingestelde sheets normaliseren
    # ---------------------------------------------------------

    def normalize_all(self):

        self.events = []

        for sheet_name in self.mappings:

            sheet_events = self.normalize_sheet(sheet_name)

            self.events.extend(sheet_events)

        # Sorteren op datum
        self.events.sort(
            key=lambda event: event["datum"]
        )

        return self.events

    # ---------------------------------------------------------
    # 9. Events als DataFrame
    # ---------------------------------------------------------

    def events_dataframe(self):

        if not self.events:
            self.normalize_all()

        if not self.events:
            return pd.DataFrame(
                columns=[
                    "Datum",
                    "Activiteit",
                    "Bron"
                ]
            )

        return pd.DataFrame([
            {
                "Datum": event["datum"],
                "Activiteit": event["activiteit"],
                "Bron": event["bron"]
            }
            for event in self.events
        ])

    # ---------------------------------------------------------
    # 10. Events tonen
    # ---------------------------------------------------------

    def print_events(self):

        if not self.events:
            self.normalize_all()

        print("\n==============================")
        print("KALENDER EVENEMENTEN")
       print("==============================")

        for event in self.events:

            print(
                event["datum"].strftime("%d-%m-%Y"),
                "|",
                event["activiteit"],
                "| bron:",
                event["bron"]
            )


# =============================================================
# AUTOMATISCHE KOLOMHERKENNING
# =============================================================

def detect_date_columns(df):
    """
    Probeert mogelijke datumkolommen te vinden.

    Dit gebruikt gewone Python/pandas-logica.
    Geen machine learning of LLM nodig.
    """

    possible_columns = []

    date_keywords = [
        "datum",
        "date",
        "startdatum",
        "start date",
        "wanneer",
        "dag"
    ]

    # Eerst kolomnamen controleren
    for column in df.columns:

        column_lower = str(column).lower().strip()

        if any(
            keyword in column_lower
            for keyword in date_keywords
        ):
            possible_columns.append(column)

    # Daarna datatype/data controleren
    for column in df.columns:

        if column in possible_columns:
            continue

        series = df[column].dropna()

        if len(series) == 0:
            continue

        converted = pd.to_datetime(
            series,
            errors="coerce",
            dayfirst=True
        )

        success_percentage = converted.notna().mean()

        if success_percentage >= 0.8:
            possible_columns.append(column)

    return possible_columns


def detect_activity_columns(df):
    """
    Probeert mogelijke activiteitskolommen te herkennen.
    """

    possible_columns = []

    activity_keywords = [
        "activiteit",
        "activity",
        "werkzaamheid",
        "werkzaamheden",
        "beschrijving",
        "description",
        "onderwerp",
        "subject",
        "event",
        "evenement",
        "taak"
    ]

    for column in df.columns:

        column_lower = str(column).lower().strip()

        if any(
            keyword in column_lower
            for keyword in activity_keywords
        ):
            possible_columns.append(column)

    return possible_columns


# =============================================================
# VOORBEELD VAN GEBRUIK
# =============================================================

def main():

    importer = ExcelCalendarImporter()

    # PAS DIT PAD AAN
    excel_file = "evenementen.xlsx"

    # Excel laden
    sheets = importer.load_excel(excel_file)

    # ---------------------------------------------------------
    # Iedere sheet bekijken
    # ---------------------------------------------------------

    for sheet_name in sheets:

        print("\n")
        print("=" * 60)
        print(f"SHEET: {sheet_name}")
        print("=" * 60)

        df = importer.sheets[sheet_name]

        print("\nVoorbeeld:")
        print(
            importer.get_preview(
                sheet_name,
                rows=5
            )
        )

        print("\nMogelijke datumkolommen:")
        date_columns = detect_date_columns(df)

        for column in date_columns:
            print(f" - {column}")

        print("\nMogelijke activiteitskolommen:")
        activity_columns = detect_activity_columns(df)

        for column in activity_columns:
            print(f" - {column}")

    # ---------------------------------------------------------
    # HIER STEL JE PER SHEET DE MAPPING IN
    #
    # Voorbeelden:
    # ---------------------------------------------------------

    # importer.set_mapping(
    #     sheet_name="Planning",
    #     date_column="Datum",
    #     activity_column="Activiteit"
    # )

    # importer.set_mapping(
    #     sheet_name="Marketing",
    #     date_column="Startdatum",
    #     activity_column="Werkzaamheid"
    # )

    # importer.set_mapping(
    #     sheet_name="Evenementen",
    #     date_column="Wanneer",
    #     activity_column="Beschrijving"
    # )

    # ---------------------------------------------------------
    # AUTOMATISCHE MAPPING
    #
    # Als precies één goede datumkolom en één goede
    # activiteitskolom worden gevonden, wordt de mapping
    # automatisch ingesteld.
    # ---------------------------------------------------------

    for sheet_name in sheets:

        df = importer.sheets[sheet_name]

        date_columns = detect_date_columns(df)
        activity_columns = detect_activity_columns(df)

        if (
            len(date_columns) == 1
            and len(activity_columns) == 1
        ):

            importer.set_mapping(
                sheet_name=sheet_name,
                date_column=date_columns[0],
                activity_column=activity_columns[0]
            )

            print(
                f"\nAutomatisch herkend voor '{sheet_name}':"
            )

            print(
                f"Datum = {date_columns[0]}"
            )

            print(
                f"Activiteit = {activity_columns[0]}"
            )

        else:

            print(
                f"\nGeen eenduidige automatische mapping "
                f"voor '{sheet_name}'."
            )

            print(
                "Stel deze handmatig in met "
                "importer.set_mapping()."
            )

    # ---------------------------------------------------------
    # Alles normaliseren
    # ---------------------------------------------------------

    events = importer.normalize_all()

    print(
        f"\nTotaal aantal geldige evenementen: "
        f"{len(events)}"
    )

    # Events tonen
    importer.print_events()

    # DataFrame maken
    calendar_df = importer.events_dataframe()

    print("\nGenormaliseerde data:")
    print(calendar_df)


if __name__ == "__main__":
    main()

SyntaxError: unexpected character after line continuation character (4241770931.py, line 46)